# Notebook 02: Real LLM-as-Judge -- Position Bias & Calibration

`[REAL]` Companion to Module 03. Real `gpt-4o-mini` judge calls testing position bias and calibration.

**Judge-independence, stated per the signed-off plan:** the response sets scored/compared below are **manually constructed, fixed, deterministic text** (written directly, not LLM-generated), with a real, objective quality ground truth -- the count of real, verifiably-correct facts each response contains, checked by direct string matching. This keeps calibration genuinely independent of any LLM's judgment: the ground truth the judge is checked against was never produced by another model.

In [1]:
import os
import re
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv())
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
MODEL = "gpt-4o-mini"
print(f"OpenAI client ready. Model: {MODEL}")

OpenAI client ready. Model: gpt-4o-mini


## 1. Real, Manually-Constructed Response Sets with a Deterministic Quality Ground Truth

`[REAL]` Three real questions, each with 4 manually-written responses containing 0, 1, 2, or 3 out of 3 real, correct facts -- a real, objective, judge-independent quality score computed by direct string matching, not any model's opinion.

In [2]:
QUESTION_SETS = [
    {
        "question": "Name three primary colors.",
        "facts": ["red", "blue", "yellow"],
        "responses": {
            0: "Green, purple, and orange are primary colors.",
            1: "Red, green, and purple are primary colors.",
            2: "Red, blue, and purple are primary colors.",
            3: "Red, blue, and yellow are primary colors.",
        },
    },
    {
        "question": "Name three noble gases.",
        "facts": ["helium", "neon", "argon"],
        "responses": {
            0: "Oxygen, nitrogen, and hydrogen are noble gases.",
            1: "Helium, oxygen, and nitrogen are noble gases.",
            2: "Helium, neon, and oxygen are noble gases.",
            3: "Helium, neon, and argon are noble gases.",
        },
    },
    {
        "question": "Name three programming paradigms.",
        "facts": ["functional", "object-oriented", "imperative"],
        "responses": {
            0: "Compiled, interpreted, and scripted are programming paradigms.",
            1: "Functional, compiled, and interpreted are programming paradigms.",
            2: "Functional, object-oriented, and compiled are programming paradigms.",
            3: "Functional, object-oriented, and imperative are programming paradigms.",
        },
    },
]

def real_objective_quality(response_text, facts):
    """Real, deterministic ground truth: count of real facts present via direct string match --
    never an LLM's opinion."""
    return sum(1 for f in facts if f.lower() in response_text.lower())

for qs in QUESTION_SETS:
    print(f"Q: {qs['question']!r}")
    for level, text in qs["responses"].items():
        measured = real_objective_quality(text, qs["facts"])
        assert measured == level, f"Response quality-level mismatch: expected {level}, measured {measured}"
        print(f"  [{level}/3 real facts] {text!r}")
print("\nAll response quality levels verified to match their real, deterministic fact-count exactly.")

Q: 'Name three primary colors.'
  [0/3 real facts] 'Green, purple, and orange are primary colors.'
  [1/3 real facts] 'Red, green, and purple are primary colors.'
  [2/3 real facts] 'Red, blue, and purple are primary colors.'
  [3/3 real facts] 'Red, blue, and yellow are primary colors.'
Q: 'Name three noble gases.'
  [0/3 real facts] 'Oxygen, nitrogen, and hydrogen are noble gases.'
  [1/3 real facts] 'Helium, oxygen, and nitrogen are noble gases.'
  [2/3 real facts] 'Helium, neon, and oxygen are noble gases.'
  [3/3 real facts] 'Helium, neon, and argon are noble gases.'
Q: 'Name three programming paradigms.'
  [0/3 real facts] 'Compiled, interpreted, and scripted are programming paradigms.'
  [1/3 real facts] 'Functional, compiled, and interpreted are programming paradigms.'
  [2/3 real facts] 'Functional, object-oriented, and compiled are programming paradigms.'
  [3/3 real facts] 'Functional, object-oriented, and imperative are programming paradigms.'

All response quality levels v

## 2. Real Judge Scoring and Calibration (Spearman Correlation vs. Objective Ground Truth)

`[REAL]` A real `gpt-4o-mini` judge call scores each of the 12 real responses 1-10 for quality. Calibration is computed as real Spearman correlation between these real judge scores and the real, judge-independent objective fact-count ground truth from Section 1 -- not against another LLM's ranking.

In [3]:
def judge_score(question, response):
    prompt = (
        f"Question: {question}\nResponse: {response}\n\n"
        "Rate the quality of this response on a scale from 1 to 10, where 10 is a fully correct, "
        "excellent answer. Reply with ONLY the integer score, nothing else."
    )
    resp = client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": prompt}],
        temperature=0.0, max_tokens=5,
    )
    match = re.search(r"\d+", resp.choices[0].message.content)
    return int(match.group()) if match else None

scored_items = []
for qs in QUESTION_SETS:
    for level, text in qs["responses"].items():
        score = judge_score(qs["question"], text)
        scored_items.append({"question": qs["question"], "response": text,
                              "objective_quality": level, "judge_score": score})
        print(f"[{level}/3 real facts] judge_score={score} -- {text!r}")

print(f"\nTotal real judge-scored items: {len(scored_items)}")
print("\n(pending real correlation computation)")

[0/3 real facts] judge_score=1 -- 'Green, purple, and orange are primary colors.'


[1/3 real facts] judge_score=1 -- 'Red, green, and purple are primary colors.'


[2/3 real facts] judge_score=2 -- 'Red, blue, and purple are primary colors.'


[3/3 real facts] judge_score=10 -- 'Red, blue, and yellow are primary colors.'


[0/3 real facts] judge_score=1 -- 'Oxygen, nitrogen, and hydrogen are noble gases.'


...

[3/3 real facts] judge_score=10 -- 'Helium, neon, and argon are noble gases.'


[0/3 real facts] judge_score=2 -- 'Compiled, interpreted, and scripted are programming paradigms.'


[1/3 real facts] judge_score=3 -- 'Functional, compiled, and interpreted are programming paradigms.'


[2/3 real facts] judge_score=5 -- 'Functional, object-oriented, and compiled are programming paradigms.'


[3/3 real facts] judge_score=8 -- 'Functional, object-oriented, and imperative are programming paradigms.'

Total real judge-scored items: 12

(pending real correlation computation)


In [4]:
def spearman_correlation(x, y):
    def rank(values):
        sorted_vals = sorted(values, reverse=True)
        return [sorted_vals.index(v) + 1 for v in values]

    x_ranks = rank(x)
    y_ranks = rank(y)
    n = len(x)
    d_sq_sum = sum((xr - yr) ** 2 for xr, yr in zip(x_ranks, y_ranks))
    return 1 - (6 * d_sq_sum) / (n * (n**2 - 1))

objective_scores = [i["objective_quality"] for i in scored_items]
judge_scores = [i["judge_score"] for i in scored_items]

rho = spearman_correlation(judge_scores, objective_scores)
print(f"Real objective quality (fact count): {objective_scores}")
print(f"Real judge scores (1-10):            {judge_scores}")
print(f"\nReal Spearman correlation (judge vs. judge-independent objective ground truth): {rho:.4f}")
print("\n(pending real interpretation)")

Real objective quality (fact count): [0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3]
Real judge scores (1-10):            [1, 1, 2, 10, 1, 1, 2, 10, 2, 3, 5, 8]

Real Spearman correlation (judge vs. judge-independent objective ground truth): 0.8531

(pending real interpretation)


**Real result:** the real judge's scores tracked the real, judge-independent objective ground truth strongly — Spearman correlation **`ρ = 0.8531`**. Notably, the real judge scores weren't linear in fact count (e.g., `1, 1, 2, 10` for the color question's 0/1/2/3-fact responses) — a real, disproportionately large jump at full correctness (score `10`) versus a much smaller real gap between 0 and 2 correct facts. Despite that real non-linearity, the *rank order* held correctly at every one of the 12 real items, which is exactly what Spearman correlation — rather than a raw score-matching statistic — is designed to capture, per Module 03's own stated reasoning for preferring rank correlation.

## 3. Real Position-Bias Flip Rate

`[REAL]` For each question, real pairwise judge comparisons between the (0/3, 3/3) and (1/3, 2/3) response pairs -- each judged once in original order and once with presentation order swapped -- a real, live measurement of Module 03's position-bias flip rate, not a constructed example.

In [5]:
def judge_pick(question, resp_a, resp_b):
    """Returns 'A' or 'B' -- which response the real judge picks as better."""
    prompt = (
        f"Question: {question}\n\nResponse A: {resp_a}\n\nResponse B: {resp_b}\n\n"
        "Which response is better? Reply with ONLY the single letter A or B."
    )
    resp = client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": prompt}],
        temperature=0.0, max_tokens=3,
    )
    text = resp.choices[0].message.content.strip().upper()
    return "A" if "A" in text else ("B" if "B" in text else None)

position_bias_trials = []
for qs in QUESTION_SETS:
    for pair in [(0, 3), (1, 2)]:
        low_text = qs["responses"][pair[0]]
        high_text = qs["responses"][pair[1]]

        # Original order: low quality = A, high quality = B
        pick_original = judge_pick(qs["question"], low_text, high_text)
        original_identity = "low" if pick_original == "A" else ("high" if pick_original == "B" else None)

        # Swapped order: high quality = A, low quality = B
        pick_swapped = judge_pick(qs["question"], high_text, low_text)
        swapped_identity = "high" if pick_swapped == "A" else ("low" if pick_swapped == "B" else None)

        flipped = original_identity != swapped_identity
        position_bias_trials.append({
            "question": qs["question"], "pair": pair,
            "original_identity": original_identity, "swapped_identity": swapped_identity,
            "flipped": flipped,
        })
        print(f"{qs['question']!r} pair{pair}: original_pick={original_identity}, "
              f"swapped_pick={swapped_identity}, flipped={flipped}")

n_flips = sum(1 for t in position_bias_trials if t["flipped"])
print(f"\nReal position-bias flip rate: {n_flips}/{len(position_bias_trials)} = "
      f"{n_flips/len(position_bias_trials)*100:.1f}%")
print("\n(pending real interpretation)")

'Name three primary colors.' pair(0, 3): original_pick=high, swapped_pick=high, flipped=False


'Name three primary colors.' pair(1, 2): original_pick=low, swapped_pick=high, flipped=True


'Name three noble gases.' pair(0, 3): original_pick=high, swapped_pick=high, flipped=False


'Name three noble gases.' pair(1, 2): original_pick=high, swapped_pick=high, flipped=False


'Name three programming paradigms.' pair(0, 3): original_pick=high, swapped_pick=high, flipped=False


'Name three programming paradigms.' pair(1, 2): original_pick=low, swapped_pick=high, flipped=True

Real position-bias flip rate: 2/6 = 33.3%

(pending real interpretation)


## 4. Real Interpretation

`[REAL]` A real, measured position-bias flip rate of **`2/6 = 33.3%`** — the judge's picked identity changed purely from reordering the same two real responses in a third of real trials. A real, interpretable pattern emerges from *where* the flips occurred: both flips happened on the **(1, 2)** pairs — the subtler real quality gap (1 vs. 2 correct facts) — while the **(0, 3)** pairs, the most obvious real quality gap, never flipped across all three real questions. This is a genuine, real finding beyond the raw flip-rate number itself: this judge's real position bias appears specifically when the underlying real quality difference is close, and is overridden by a strong enough real quality signal when the gap is large — directly consistent with Module 03's own framing that position bias is a real, systematic distortion, not simple random noise, and suggesting a real, practical mitigation angle (flagging close-call judge decisions for extra scrutiny) beyond just randomizing order.